# Single Subject Simulation

We need the empirical distributions for this one subject:

In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/Users/braydenchien/Desktop/Enkavilab/DDM/1ms_trial_data.csv')
df['RT'] = df['RT']*1000 # adjustment for RT
df['fixation'] = df['fixation'].apply(literal_eval)

sub_id = 301
sub_df = df.loc[df['sub_id'] == 301] # No dropped trials, second batch had less errors overall, more trials

Let's build a function to capture the corrected empirical distributions for a subject that discard the top 5% of durations across all empirical distributions lists.

In [ ]:
from simulation import get_corrected_empirical_distributions

value_diffs = np.arange(-4, 4.25, 0.25)
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_empirical_distributions(
    sub_df, 
    value_diffs, 
    legend=legend, 
    fixation_col=fixation_col, 
    left_value_col=left_value_col, 
    right_value_col=right_value_col, 
    cutoff=0.95
)

Instead of using create trials from `simulation.py`, I will use `odd_sub_df` to act as a stand in. We'll need to clean things up and then generate fixations.

In [ ]:
from simulation import generate_fixations

dt = 0.001

trials = sub_df.loc[sub_df['trial'] % 2 == 1].loc[:, ['avgWTP_left', 'avgWTP_right']]
trials['fixation'] = None

rows = []
for idx, r in trials.iterrows():
    fx = generate_fixations(dt, r.avgWTP_left - r.avgWTP_right, empirical_distributions)
    if fx is not None:
        rows.append((idx, fx))

trials_clean = trials.loc[[i for i, _ in rows]].copy()
trials_clean['fixation'] = [fx for _, fx in rows]
my_trials = trials_clean.to_dict(orient="records")

## Inspect single subject model-free analysis

In [ ]:
from mfa import plot_basic_psychometrics, plot_fixation_properties
from pyddm.preprocessing.dataset import rasterize_data

sub_df["choice"] = sub_df["choice"].map({"left": 0, "right": 1})

subject_rasterized = rasterize_data(sub_df, subject_col='sub_id',trial_col='trial',seq_col='fixation')

subject_rasterized['fix_dur'] = subject_rasterized.apply(
    lambda r: r['fix_end'] - r['fix_start'],
    axis=1
)

subject_rasterized['fix_num'] = (
    subject_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount() + 1
)

subject_rasterized['fix_num_rev'] = (
    subject_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount(ascending=False) + 1
)

plot_fixation_properties(subject_rasterized)
plot_basic_psychometrics(subject_rasterized)

## Simulate trials

In [ ]:
from simulation import simulate

seed = 42
model_conditions = {'drift_rate': 0.7, 'theta': 0.5, 'noise': 0.3}

results_df = simulate(dt, model_conditions, my_trials, seed=seed, save_results=False)

results_df['sub_id'] = str(sub_id) + '_sim'
results_df['trial'] = range(1, len(trials_clean) + 1)
results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])
print(np.average(results_df.loc[:, "RT"]))
results_df.head()

## Rerasterize simulated data

The following should be turned into a function and reworked into the pipeline. We need to create fix_num, fix_num_rev, fix_dur.

In [ ]:
from pyddm.preprocessing.dataset import rasterize_data

trials_rasterized = rasterize_data(results_df, subject_col='sub_id',trial_col='trial',seq_col='fix_sequence')
trials_rasterized['fix_dur'] = trials_rasterized.apply(
    lambda r: r['fix_end'] - r['fix_start'],
    axis=1
)

trials_rasterized['fix_num'] = (
    trials_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount() + 1
)

trials_rasterized['fix_num_rev'] = (
    trials_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount(ascending=False) + 1
)
trials_rasterized.head()

In [ ]:
from mfa import plot_basic_psychometrics, plot_fixation_properties

plot_basic_psychometrics(trials_rasterized)
plot_fixation_properties(trials_rasterized)

There are two reasons why the model-free analyses graphs may be different. First, the net fixation duration by relative value difference graph heavily depends on in-sample relative value difference combos. By only isolating one subject's empirical trials, we lose the contributions of many relative value difference combos as each subject have uneven combos hand selected from survey data. Second, and probably still a main issue, perhaps parameters used to estimate the data are unstable. As suggested by the `simple_grid_search` notebook, parameter recovery heavily depends on correct calculations for RT, scalings for model dt, and noise. These confounding factors are present in the simulation pipeline.